In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Conv1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# =========================
# Configuration and constants
# =========================
ROOT_DIR = 'All_10person_Cycles'
MAX_SEQUENCE_LENGTH = 180
TRAIN_RATIO = 0.8
VALID_LABELS = ['back', 'front', 'normal', 'side']
FEATURE_COLUMNS = [
    'IMU101_v0', 'IMU101_v1', 
    'IMU103_v0', 'IMU103_v1', 
    'IMU104_v0', 'IMU104_v1', 'IMU301_v0', 'IMU301_v1', 
    #'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6',
    #'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]

# Model hyperparameters
UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32

def load_and_split_data():
    """Load cycles, select features, pad/truncate, encode labels, and split."""
    X_data, y_labels, cycle_ids = [], [], []

    if not os.path.exists(ROOT_DIR):
        return None, None, None, None, 0, 0, None

    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                file_path = os.path.join(root, filename)

                # Parse label and cycle id from filename
                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cycle_id_str = raw_id_part.split('_')[-1]

                if label not in VALID_LABELS or not cycle_id_str.isdigit():
                    continue

                try:
                    df = pd.read_csv(file_path)
                except Exception:
                    continue

                # Feature selection
                if any(col not in df.columns for col in FEATURE_COLUMNS):
                    continue
                df = df[FEATURE_COLUMNS]
                cycle_features = df.values
                if cycle_features.shape[0] == 0 or cycle_features.shape[1] == 0:
                    continue

                # Pad/truncate to fixed length
                if cycle_features.shape[0] < MAX_SEQUENCE_LENGTH:
                    pad_len = MAX_SEQUENCE_LENGTH - cycle_features.shape[0]
                    padded = np.pad(cycle_features, ((0, pad_len), (0, 0)), 'constant', constant_values=0)
                else:
                    padded = cycle_features[:MAX_SEQUENCE_LENGTH, :]

                X_data.append(padded)
                y_labels.append(label)
                cycle_ids.append(file_path)

    if not X_data:
        return None, None, None, None, 0, 0, None

    X_full = np.array(X_data)
    y_full = np.array(y_labels)
    ids_full = np.array(cycle_ids)

    # Encode labels
    le = LabelEncoder()
    y_int = le.fit_transform(y_full)
    y_encoded = to_categorical(y_int)
    class_names = le.classes_

    num_features = X_full.shape[2]
    num_classes = y_encoded.shape[1]

    # Train/test split by unique file paths
    unique_ids = np.unique(ids_full)
    train_ids, test_ids = train_test_split(unique_ids, train_size=TRAIN_RATIO, random_state=42, shuffle=True)
    train_idx = np.where(np.isin(ids_full, train_ids))[0]
    test_idx = np.where(np.isin(ids_full, test_ids))[0]

    X_train, X_test = X_full[train_idx], X_full[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    return X_train, X_test, y_train, y_test, num_features, num_classes, class_names

def build_model(model_type, sequence_length, num_features, num_classes):
    """Build and compile a model of the requested type: 'lstm', 'gru', or 'cnn'."""
    if model_type == 'lstm':
        model = Sequential([
            LSTM(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='LSTM'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'gru':
        model = Sequential([
            GRU(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='GRU'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'cnn':
        model = Sequential([
            # Convolution over time steps
            Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=(sequence_length, num_features)),
            Dropout(DROPOUT_RATE),
            Conv1D(filters=64, kernel_size=5, activation='relu'),
            GlobalAveragePooling1D(),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def train_and_compare():
    """Train and evaluate multiple architectures on the same split."""
    X_train, X_test, y_train, y_test, num_features, num_classes, class_names = load_and_split_data()
    if X_train is None or X_train.shape[0] < 1:
        print("Insufficient data; training aborted.")
        return

    sequence_length = X_train.shape[1]
    model_types = ['lstm', 'gru', 'cnn']

    for mtype in model_types:
        print(f"\n=== Training {mtype.upper()} model ===")
        model = build_model(mtype, sequence_length, num_features, num_classes)

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
            ModelCheckpoint(f'best_{mtype}_IMU.weights.h5', monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=0)
        ]

        history = model.fit(
            X_train, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=(X_test, y_test),
            callbacks=callbacks,
            verbose=1
        )

        # Load best weights and evaluate
        try:
            model.load_weights(f'best_{mtype}_IMU.weights.h5')
        except Exception as e:
            print(f"Warning: Could not load best weights for {mtype}: {e}")

        loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
        print(f"{mtype.upper()} Test Loss: {loss:.4f} | Test Accuracy: {accuracy:.4f}")

        # Classification report
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        y_true = np.argmax(y_test, axis=1)
        print(f"\n{mtype.upper()} Classification Report:")
        print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

if __name__ == '__main__':
    tf.get_logger().setLevel('INFO')
    train_and_compare()

2026-01-10 13:37:05.593902: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768070225.607708 2858481 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768070225.612056 2858481 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768070225.623771 2858481 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768070225.623787 2858481 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768070225.623790 2858481 computation_placer.cc:177] computation placer alr


=== Training LSTM model ===


I0000 00:00:1768070245.898103 2858481 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 991 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1768070245.902896 2858481 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 46147 MB memory:  -> device: 1, name: NVIDIA RTX A6000, pci bus id: 0000:25:00.0, compute capability: 8.6
I0000 00:00:1768070245.904333 2858481 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 40243 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1768070245.905599 2858481 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 29994 MB memory:  -> device: 3, name: NVIDIA RTX A6000, pci bus id: 0000:c1:00.0, compute capability: 8.6
/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarni

Epoch 1/25


I0000 00:00:1768070247.844487 2859320 cuda_dnn.cc:529] Loaded cuDNN version 90701


...

130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.2587 - loss: 1.4081 - val_accuracy: 0.2737 - val_loss: 1.3744
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.2633 - loss: 1.3746 - val_accuracy: 0.2582 - val_loss: 1.3897
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.2945 - loss: 1.3677 - val_accuracy: 0.2727 - val_loss: 1.3814
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.3339 - loss: 1.3273 - val_accuracy: 0.5193 - val_loss: 1.0733
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.4857 - loss: 1.0599 - val_accuracy: 0.5455 - val_loss: 0.9538
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5463 - loss: 0.9185 - val_accuracy: 0.6044 - val_loss: 0.8678
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6230 - loss: 0.8254 - val_accuracy: 0.6925 - val_loss: 0.7275
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6958 - loss: 0.7135 - val_accuracy: 0.7321 - val

/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1768070298.408671 2859320 service.cc:152] XLA service 0xadeb7b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768070298.408729 2859320 service.cc:160]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1768070298.408735 2859320 service.cc:160]   StreamExecutor device (1): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1768070298.408737 2859320 service.cc:160]   StreamExecutor device (2): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1768070298.408738 2859320 service.cc:160]   StreamExecutor device (3): NVIDIA RTX A6000, Compute Capabi

107/130 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4100 - loss: 4.1311  

I0000 00:00:1768070300.314908 2859320 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


130/130 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.4322 - loss: 3.7068 - val_accuracy: 0.6973 - val_loss: 0.7435
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7159 - loss: 0.6920 - val_accuracy: 0.7911 - val_loss: 0.5546
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7947 - loss: 0.5211 - val_accuracy: 0.8733 - val_loss: 0.3487
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8510 - loss: 0.3906 - val_accuracy: 0.9217 - val_loss: 0.2975
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8777 - loss: 0.3297 - val_accuracy: 0.9197 - val_loss: 0.2432
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8937 - loss: 0.2865 - val_accuracy: 0.9545 - val_loss: 0.1962
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9044 - loss: 0.2577 - val_accuracy: 0.9333 - val_loss: 0.1906
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9105 - loss: 0.2613 - val_accuracy: 0.9410 - val